In [1]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.streaming import StreamingQuery
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import col, rand

### Create Spark Session

In this cell, we will initialize the `SparkSession`, which is the entry point to programming Spark with the Dataset and DataFrame API. We set the application name and call `getOrCreate()` to ensure a session is available for our data processing tasks.

In [2]:
def create_spark_session(
    s3_endpoint:str,
    s3_user:str, 
    s3_pass:str,
    s3_bucket:str) -> SparkSession:
    """
    Creates a SparkSession with Delta Lake and S3/MinIO configurations.
    """
    builder:SparkSession.Builder = SparkSession.builder \
        .master("local[*]") \
        .appName("Streaming-Simulation") \
        .config("spark.driver.memory", "4g") \
        .config("spark.executor.memory", "4g") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .config("spark.hadoop.fs.s3a.endpoint", s3_endpoint) \
        .config("spark.hadoop.fs.s3a.access.key", s3_user) \
        .config("spark.hadoop.fs.s3a.secret.key", s3_pass) \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
        .config("spark.sql.warehouse.dir", f"s3a://{s3_bucket}")

    return configure_spark_with_delta_pip(builder, extra_packages=[
        "org.apache.hadoop:hadoop-aws:3.3.4",
        "com.amazonaws:aws-java-sdk-bundle:1.12.262"
    ]).getOrCreate()

# Define the S3-Bucket name
bucket_name:str = "warehouse"
stream_data_name:str = "stream_data"
stream_instance:int = "4"
processing_interval:str = "60 seconds"
max_records_per_file:int = 1000000

# Fetch spark session
spark:SparkSession = create_spark_session(
    s3_endpoint="http://127.0.0.1:9000",
    s3_user="minioadmin",
    s3_pass="minioadmin",
    s3_bucket="warehouse"
)
spark.sparkContext.setLogLevel("WARN")

26/01/26 17:29:13 WARN Utils: Your hostname, DESKTOP-L43ICML resolves to a loopback address: 127.0.1.1; using 172.30.231.238 instead (on interface eth0)
26/01/26 17:29:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/castroaj/dev/git/delta-lake-demo/.venv/lib/python3.8/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/castroaj/.ivy2/cache
The jars for the packages stored in: /home/castroaj/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-50e02479-dde1-464a-bb15-798e62579aa0;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 325ms :: artifacts dl 15ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#ant

### Fetch Data-Stream

Additionally lets fetch a data stream of records, moving at 5 records per second

In [3]:
def fetch_stream_datasource(records_per_second:int) -> DataFrame:
    """
    Fetch a stream datasource of `x` records per second
    """
    stream_df:DataFrame = spark.readStream.format("rate").option("rowsPerSecond", records_per_second).load()
    return stream_df

# Fetch stream dataframe
stream_df:DataFrame = fetch_stream_datasource(records_per_second=25000)

26/01/26 17:29:19 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


### Stream Random Values into S3

Setup a streaming `DataFrame` that generates random `id` and `rand-val`, and then processes them on the configured interval

In [ ]:
# Simulate random stream data
processed_df:DataFrame = stream_df.select(
    col("timestamp").alias("event_time"),
    (rand() * 100).cast("int").alias("id"),
    (rand() * 30 + 20).alias("rand-val")
)

# Create a streaming query that writes to the s3 bucket
query:StreamingQuery = (processed_df.coalesce(1).writeStream
    .format("delta")
    .outputMode("append")
    .option("clusteringColumns", "id")
    .option("checkpointLocation", f"s3a://{bucket_name}/_checkpoints/{stream_data_name}_{stream_instance}")
    .option("maxRecordsPerFile", max_records_per_file)
    .trigger(processingTime=processing_interval)
    .start(f"s3a://{bucket_name}/{stream_data_name}_{stream_instance}")
)
print(f"Streaming has started to {bucket_name}_{stream_instance}... Data is being written to S3")
query.awaitTermination(timeout=processing_interval)

26/01/26 17:29:24 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Streaming has started to warehouse_4... Data is being written to S3


26/01/26 17:29:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


False

### Query Data

Query the data from the `S3 Bucket` by loading the table as a delta table

In [23]:
# Read the table normally - it will show new data every time you run it
df:DataFrame = spark.read.format("delta").load(f"s3a://{bucket_name}/{stream_data_name}_{stream_instance}")
print(df.count())
print(f"IsActive={query.isActive}")
# df.show(df.count())

8375000
IsActive=False


### End Streaming Job

The following will kill-streaming job, as it will be running continuously in the background

In [22]:
# Kill the streaming query
query.stop()